# 1. Environment Setup
Install the necessary libraries for fine-tuning with TRL, PEFT, and BitsAndBytes.

In [ ]:
!pip install -q trl transformers accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets

In [ ]:
!pip install -q --upgrade wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 71.3 MB/s eta 0:00:00


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import wandb

# 2. Data Loading and Authentication
Loading the MedQuad dataset and setting up Hugging Face and Weights & Biases credentials.

In [ ]:
# Model from Hugging Face hub
base_model = "meta-llama/Llama-3.2-3B-Instruct"
# Fine-tuned model
new_model = "medquad_assistant_v1"

In [ ]:
# Load the dataset directly from Hugging Face hub
dataset = load_dataset('keivalya/MedQuad-MedicalQnADataset', split='train')
print('Dataset loaded successfully!')
print(dataset[0])

README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

Dataset loaded successfully!
{'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?', 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}


In [ ]:
len(dataset)

16407

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [ ]:
# Log in to Weights & Biases
wandb_api_key = user_secrets.get_secret('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = "medquad_chatbot"
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sumanta049 (sumanta049-jx) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
wandb.init(project="medquad_assistant_v1", name="medquad_v1")

wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260323_131203-1vzv09jg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run medquad_v1
wandb: ⭐️ View project at https://wandb.ai/sumanta049-jx/medquad_assistant_v1
wandb: 🚀 View run at https://wandb.ai/sumanta049-jx/medquad_assistant_v1/runs/1vzv09jg


# 3. Model and Quantization Configuration
Defining hyper-parameters and initializing the Llama 3.2 model with 4-bit quantization.

In [ ]:
# Hyper-parameters - overall
EPOCHS = 1
BATCH_SIZE = 2 #32
MAX_SEQUENCE_LENGTH = 256
GRADIENT_ACCUMULATION_STEPS = 4

# Hyper-parameters - QLoRA
QUANT_4_BIT = True
LORA_R = 32
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

# Tracking
LOG_STEPS = 100
SAVE_STEPS = 500
LOG_TO_WANDB = True

In [ ]:
compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map="auto",     # {"": 0}
    torch_dtype=torch.float16
)
model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
# LoRA Parameters
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

# 4. Fine-Tuning with SFTTrainer
Formatting the dataset into the Llama 3 format and starting the Supervised Fine-Tuning process.

In [ ]:
# Training parameters
train_parameters = SFTConfig(
    output_dir="medquad_assistant_v1",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    #group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else "tensorboard",
    run_name="medquad_v1",
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id="medquad_assistant",
    hub_private_repo=True,
    #eval_strategy="steps",
    #eval_steps=SAVE_STEPS
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# reate the Llama 3.2 chat template format
def format_llama3(example):
    system_prompt = "You are a helpful and accurate medical assistant."
    user_msg = example["Question"]
    model_msg = example["Answer"]

    # Llama 3.2 specific token structure
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_prompt}"
        "<|eot_id|>"

        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_msg}"
        "<|eot_id|>"

        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{model_msg}"
        "<|eot_id|>"
    )

    return {"text": text}

dataset = dataset.map(format_llama3, batched=False)

dataset = dataset.filter(lambda x: x["text"] is not None and isinstance(x["text"], str))

Map:   0%|          | 0/16407 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16407 [00:00<?, ? examples/s]

In [ ]:
#for 1000 rows only
# #Shuffle the data and select a smaller chunk (eg: 200 rows)

# num_samples = 100
# dataset = dataset.shuffle(seed=42).select(range(num_samples))
# print(f"Training on a subset of {len(dataset)} examples.")

In [ ]:
# Initialize the Trainer

fine_tuning = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_parameters,
    args=train_parameters
    #processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/16407 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16407 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16407 [00:00<?, ? examples/s]

In [ ]:
fine_tuning.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
100,1.441533
200,1.153623
300,1.106069
400,1.082553
500,1.062340
600,1.044867
700,1.043680
800,1.027968
900,1.046289
1000,1.045582


TrainOutput(global_step=2051, training_loss=1.0524173496642502, metrics={'train_runtime': 9731.9663, 'train_samples_per_second': 1.686, 'train_steps_per_second': 0.211, 'total_flos': 6.542084901432115e+16, 'train_loss': 1.0524173496642502})

# 5. Export and Inference
Merging LoRA weights with the base model and pushing the final model to the Hugging Face Hub.

In [ ]:
fine_tuning.model.push_to_hub("medquad_assistant_v1")
print("Saved to the hub: medquad_assistant_v1")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved to the hub: medquad_assistant_v1


In [ ]:
if LOG_TO_WANDB:
  wandb.finish()

wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log; uploading config.yaml
wandb: uploading summary, console lines 3-3
wandb: 
wandb: Run history:
wandb:             train/entropy █▄▃▃▃▂▂▂▂▂▁▂▂▂▂▁▂▂▂▂▁
wandb:               train/epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
wandb:         train/global_step ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
wandb:           train/grad_norm ▆▅▂▆▂▃▇█▂▅▂▁▆▄▃▄▅▅▆▄
wandb:       train/learning_rate ███▇▇▇▆▆▅▅▄▄▃▃▂▂▁▁▁▁
wandb:                train/loss █▄▃▂▂▂▂▂▂▂▁▁▁▁▂▁▁▁▁▁
wandb: train/mean_token_accuracy ▁▅▆▆▆▇▇▇▇▇█▇▇▇▇█▇▇▇▇█
wandb:          train/num_tokens ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
wandb: 
wandb: Run summary:
wandb:                total_flos 6.542084901432115e+16
wandb:             train/entropy 1.03668
wandb:               train/epoch 1
wandb:         train/global_step 2051
wandb:           train/grad_norm 0.72266
wandb:       train/learning_rate 0.0
wandb:                train/loss 0.99981
wandb: train/m

In [ ]:
fine_tuning.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

('medquad_assistant_v1/tokenizer_config.json',
 'medquad_assistant_v1/chat_template.jinja',
 'medquad_assistant_v1/tokenizer.json')

In [ ]:
logging.set_verbosity(logging.CRITICAL)
model.eval()

prompt = "What are the common symptoms of back pain?"
system_prompt = "You are a helpful and accurate medical assistant."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt},
]
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
 )


eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
stop_ids = [tokenizer.eos_token_id]
if eot_id is not None and eot_id != tokenizer.eos_token_id:
    stop_ids.append(eot_id)

pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
 )

result = pipe(
    prompt_text,
    max_new_tokens=120,
    do_sample=False,
    repetition_penalty=1.05,
    no_repeat_ngram_size=4,
    eos_token_id=stop_ids,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
 )

answer_only = result[0]["generated_text"].strip()
print(answer_only)

The most common symptom of back pain is pain in the lower back, which is called the lumbar region. The pain may be sharp or dull, and it can range from mild to severe. Back pain can also cause numbness, tingling, weakness, and muscle spasms. In some cases, back pain can radiate to other parts of the body, such as the arms, legs, or buttocks.


In [ ]:
model,tokenizer

(LlamaForCausalLM(
   (model): LlamaModel(
     (embed_tokens): Embedding(128256, 3072)
     (layers): ModuleList(
       (0-27): 28 x LlamaDecoderLayer(
         (self_attn): LlamaAttention(
           (q_proj): lora.Linear4bit(
             (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
             (lora_dropout): ModuleDict(
               (default): Dropout(p=0.1, inplace=False)
             )
             (lora_A): ModuleDict(
               (default): Linear(in_features=3072, out_features=32, bias=False)
             )
             (lora_B): ModuleDict(
               (default): Linear(in_features=32, out_features=3072, bias=False)
             )
             (lora_embedding_A): ParameterDict()
             (lora_embedding_B): ParameterDict()
             (lora_magnitude_vector): ModuleDict()
           )
           (k_proj): lora.Linear4bit(
             (base_layer): Linear4bit(in_features=3072, out_features=1024, bias=False)
             (lora_dropo

In [ ]:
load_model = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

model = PeftModel.from_pretrained(load_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it safely
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
# Only use existing tokens to prevent embedding size mismatches
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
model.push_to_hub(new_model)
tokenizer.push_to_hub(new_model)

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Lucifer049/medquad_assistant_v1/commit/54bcf1c8a88a57ad465cf25922e3d2cb56c755b1', commit_message='Upload tokenizer', commit_description='', oid='54bcf1c8a88a57ad465cf25922e3d2cb56c755b1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Lucifer049/medquad_assistant_v1', endpoint='https://huggingface.co', repo_type='model', repo_id='Lucifer049/medquad_assistant_v1'), pr_revision=None, pr_num=None)